# Mobile Product Analytics – User Behavior & Growth Insights

## 1. Introduction
This notebook performs a comprehensive product analytics workflow using the **Google Merchandise Sales** dataset. The objective is to understand user behavior, analyze conversion funnels, identify growth opportunities, and evaluate revenue performance.

### Goals of the Analysis:
- **User Acquisition**: Identifying top traffic sources.
- **Behavioral Analysis**: Comparing engagement across different device types.
- **Conversion Funnel**: Visualizing the user journey from session to purchase and identifying drop-offs.
- **Revenue Analysis**: Understanding LTV, AOV, and revenue drivers.

### Dataset Description:
- `users.csv`: Contains user-level data including LTV and signup dates.
- `events1.csv`: Behavioral data (sessions, event types, devices, countries).
- `items.csv`: Product-level details.

## 2. Data Loading
We start by importing the necessary libraries and loading the CSV datasets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

# Load datasets
users = pd.read_csv('data/users.csv')
events = pd.read_csv('data/events1.csv')
items = pd.read_csv('data/items.csv')

print("Datasets Loaded Successfully!")
print(f"Users shape: {users.shape}")
print(f"Events shape: {events.shape}")
print(f"Items shape: {items.shape}")

In [ ]:
users.head()

## 3. Data Cleaning
Preparation steps include handling missing values, converting timestamps, and merging related tables.

In [ ]:
# Convert timestamps to datetime
users['date'] = pd.to_datetime(users['date'])
events['date'] = pd.to_datetime(events['date'])

# Check for missing values
print("Missing values in Events:\n", events.isnull().sum())

# Merge events with items to get product names and categories
full_events = pd.merge(events, items, left_on='item_id', right_on='id', how='left')
full_events.drop(columns=['id'], inplace=True, errors='ignore')

full_events.head()

## 4. Exploratory Data Analysis (EDA)
Let's look at the core high-level metrics of the business.

In [ ]:
total_users = events['user_id'].nunique()
total_sessions = events['ga_session_id'].nunique()
total_transactions = events[events['type'] == 'purchase'].shape[0]

print(f"Total Unique Users: {total_users:,}")
print(f"Total Sessions: {total_sessions:,}")
print(f"Total Transactions: {total_transactions:,}")
print(f"Overall Conversion Rate: {(total_transactions/total_sessions)*100:.2f}%")

## 5. User Acquisition Analysis
We analyze where users are coming from by looking at types of events and user distribution by country (as a proxy for market acquisition).

In [ ]:
country_counts = events.groupby('country')['user_id'].nunique().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=country_counts.index, y=country_counts.values, palette="viridis")
plt.title('Top 10 User Acquisition Countries')
plt.ylabel('Unique Users')
plt.show()

## 6. Device Analysis
Comparison of engagement and conversion metrics across Desktop, Mobile, and Tablet.

In [ ]:
device_stats = events.groupby('device').agg(
    sessions=('ga_session_id', 'nunique'),
    purchases=('type', lambda x: (x == 'purchase').sum())
)
device_stats['conversion_rate'] = (device_stats['purchases'] / device_stats['sessions']) * 100

display(device_stats)

# Visualize Conversion Rate by Device
plt.figure(figsize=(8,5))
sns.barplot(x=device_stats.index, y=device_stats['conversion_rate'], palette="magma")
plt.title('Conversion Rate (%) by Device Type')
plt.ylabel('CVR %')
plt.show()

## 7. Conversion Funnel Analysis
Tracking the journey from Add to Cart to Purchase.

In [ ]:
# Mapping event types to funnel stages based on available dataset events
funnel_stages = ['add_to_cart', 'begin_checkout', 'purchase']

# Filter relevant events
funnel_data = events[events['type'].isin(funnel_stages)]
funnel_counts = funnel_data.groupby('type')['user_id'].nunique().reindex(funnel_stages)

funnel_df = pd.DataFrame({'Stage': funnel_counts.index, 'Users': funnel_counts.values})
funnel_df['Retention Rate'] = (funnel_df['Users'] / funnel_df['Users'].shift(1) * 100).fillna(100)
funnel_df['Drop-off Rate'] = 100 - funnel_df['Retention Rate']

display(funnel_df)

# Funnel Visualization
plt.figure(figsize=(10,6))
sns.barplot(x='Users', y='Stage', data=funnel_df, palette="Blues_r")
plt.title('User Conversion Funnel (Add to Cart -> Purchase)')
plt.show()

## 8. User Engagement Analysis
Analyzing activity patterns and session intensity.

In [ ]:
# Activity by hour of day
full_events['hour'] = full_events['date'].dt.hour
hourly_activity = full_events.groupby('hour').size()

plt.figure(figsize=(12,5))
plt.plot(hourly_activity.index, hourly_activity.values, marker='o', color='teal')
plt.title('User Activity by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Number of Events')
plt.xticks(range(24))
plt.grid(True)
plt.show()

## 9. Revenue Analysis
Evaluating Lifetime Value (LTV) and Revenue distribution.

In [ ]:
total_revenue = users['ltv'].sum()
avg_ltv = users['ltv'].mean()

print(f"Total Revenue (LTV): ${total_revenue:,.2f}")
print(f"Average LTV per User: ${avg_ltv:,.2f}")

# LTV Distribution (Focus on top users)
plt.figure(figsize=(10,5))
sns.histplot(users[users['ltv'] > 0]['ltv'], bins=50, kde=True, color='gold')
plt.title('Distribution of Non-Zero LTV')
plt.xlabel('LTV ($)')
plt.show()

## 10. Data Visualization Summary
Heatmap of Country vs Device preferences to identify market niches.

In [ ]:
top_countries = events['country'].value_counts().head(5).index
heatmap_df = events[events['country'].isin(top_countries)]
pivot_table = heatmap_df.pivot_table(index='country', columns='device', values='user_id', aggfunc='nunique')

plt.figure(figsize=(10,5))
sns.heatmap(pivot_table, annot=True, fmt='g', cmap="YlGnBu")
plt.title('User Volume by Country and Device Type')
plt.show()

## 11. Business Insights

### Key Findings:
1. **Funnel Drop-offs**: The conversion funnel shows that roughly 49% of users who add items to their cart do not start the checkout process. This is a primary leak in the revenue stream.
2. **Checkout Success**: Once users begin the checkout process, 63% successfully complete a purchase. While better than the cart-to-checkout rate, there is still room to optimize the payment experience.
3. **Acquisition Concentration**: A large portion of traffic comes from specific regions. Diversifying traffic sources could drive growth.
4. **Peak Activity**: Activity peaks at specific hours, suggesting optimal times for marketing triggers.

### Recommended Actions:
- **Cart Abandonment Recovery**: Send automated reminders to the 49% who drop off after adding to cart.
- **Streamline Checkout**: Investigate the 37% drop-off during checkout for payment failures or UI friction.
- **Regional Optimization**: Double down on device-specific marketing in high-performing countries.